# Evaluation objects and pipelines

In [ ]:
%load_ext autoreload
%autoreload 2

import open3d # open3d can occasionally cause issues during imports; importing it first can help alleviate that
import numpy as np
from rdkit import Chem
from shepherd_score.conformer_generation import embed_conformer_from_smiles

from shepherd_score.evaluations.evaluate import ConfEval, UnconditionalEvalPipeline
from shepherd_score.evaluations.evaluate import ConsistencyEvalPipeline, ConditionalEvalPipeline

## Conformer evaluation base class

The base class used to evaluate conformer validity and get 2D graph properties is `ConfEval`. Other evaluation classes (other than docking) inherit from `ConfEval` and the related pipelines utilize these objects. 

Let's run a small experiment where the MMFF94-relaxed molecule is the "generated" molecule -- represent it as an atomic point cloud.

In [ ]:
rdkit_mol = embed_conformer_from_smiles('c1Cc2ccc(Cl)cc2C(=O)c1c3cc(N1nnc2cc(C)c(Cl)cc2c1=O)ccc3', MMFF_optimize=True)

# get the atomic numbers as an array and the positions of the atoms
atoms = np.array([a.GetAtomicNum() for a in rdkit_mol.GetAtoms()])
positions = rdkit_mol.GetConformer().GetPositions()

In [ ]:
conf_eval = ConfEval(atoms, positions, solvent='water') # solvent = None if gas phase

In [ ]:
conf_eval.to_pandas() # show the attributes as a pandas Series

## Conformer evaluation pipelines

Since typically multiple molecules are generated and all need to be evaluated, some pipeline classes used.

### Unconditional evaluation
The `UnconditionalEvalPipeline` simply iterates over all the generated molecules with `ConfEval` and stores the full evaluation.

Let's generate a few test molecules and embed them with RDKit ETKDG. We prepare them for the necessary inputs: a list of tuples containing each the molecule's corresponding atoms' atomic numbers and positions as numpy arrays.

In [ ]:
smiles_ls = ['CC', 'CCC', 'CCCC']*100
test_mols = [embed_conformer_from_smiles(smi, MMFF_optimize=False) for smi in smiles_ls]

generated_mols = []
for m in test_mols:
    generated_mols.append(
        (np.array([a.GetAtomicNum() for a in m.GetAtoms()]), m.GetConformer().GetPositions())
    )

Initialize and run the pipeline.

In [ ]:
uncond_pipe = UnconditionalEvalPipeline(generated_mols=generated_mols, solvent='water')

In [ ]:
uncond_pipe.evaluate(verbose=True, num_processes=4, num_workers=1)

In [ ]:
uncond_pipe.evaluate(verbose=True, num_processes=1, num_workers=8)

In [ ]:
properties_df, global_attr = uncond_pipe.to_pandas()

In [ ]:
global_attr.head()

In [ ]:
properties_df

### Consistency evaluation

This is used to evaluate if the jointly generated interaction profiles correspond to the true interaction profile of the generated molecule. The `ConsistencyEvalPipeline` simply iterates over all the generated molecules with the `ConsistencyEval` class and stores the full evaluation. In addition to the properties calculated by `ConfEval` it also does score-based alignment so it is a slower operation.

In [ ]:
from shepherd_score.container import Molecule

Prepare the inputs. We pretend that the test smiles are "generated" molecules with their corresponding interaction profiles. `ConsistencyEvalPipeline` expects this format for the inputs.

In [ ]:
smiles_ls = ['CC', 'CCC', 'CCCC']*33
test_mols = [embed_conformer_from_smiles(smi, MMFF_optimize=True) for smi in smiles_ls]

generated_mols = []
generated_surf_points = []
generated_surf_esp = []
generated_pharm_feats = []
for m in test_mols:
    generated_mols.append(
        (np.array([a.GetAtomicNum() for a in m.GetAtoms()]), m.GetConformer().GetPositions())
    )
    # Generate and store each interaction profile as if they were generated.
    # Notably, we use MMFF94 partial charges and ConsistencyEvalPipeline
    #  will compare the ESP to xTB generated partial charges
    molec = Molecule(m, num_surf_points=200, probe_radius=1.2, partial_charges=None, pharm_multi_vector=False)
    generated_surf_points.append(molec.surf_pos)
    generated_surf_esp.append(molec.surf_esp)
    generated_pharm_feats.append(
        (molec.pharm_types, molec.pharm_ancs, molec.pharm_vecs)
    )

Initialize and run the pipeline

In [ ]:
consis_eval = ConsistencyEvalPipeline(
    generated_mols = generated_mols,
    generated_surf_points = generated_surf_points,
    generated_surf_esp = generated_surf_esp,
    generated_pharm_feats = generated_pharm_feats,
    probe_radius=1.2,
    pharm_multi_vector=False,
    solvent=None
)

In [ ]:
consis_eval.evaluate(num_workers=1, verbose=True)

In [ ]:
consis_eval.evaluate(num_workers=10, verbose=True)

You can view the saved attributes and properties as a pandas Series for the global (whole set) attributes and a DataFrame for per-sample properties.

In [ ]:
properties_df_consis, global_attr_consis = consis_eval.to_pandas()

In [ ]:
global_attr_consis.head()

In [ ]:
properties_df_consis

### Conditional evaluation

This is used to evaluate if the generated is similar (based on `shepherd_score` 3D scoring functions) to the target/reference molecule. The `ConditionalEvalPipeline` simply iterates over all the generated molecules with the `ConditionalEval` class and stores the full evaluation. In addition to the properties calculated by `ConfEval` it also does score-based alignment so it is a slower operation.

In [ ]:
rdkit_mol = embed_conformer_from_smiles('c1Cc2ccc(Cl)cc2C(=O)c1c3cc(N1nnc2cc(C)c(Cl)cc2c1=O)ccc3', MMFF_optimize=True)

# Again using MMFF94 partial charges
ref_molec = Molecule(rdkit_mol, num_surf_points=200, probe_radius=1.2, pharm_multi_vector=False)

In [ ]:
cond_pipe = ConditionalEvalPipeline(ref_molec, generated_mols=generated_mols,
                                    condition='all', num_surf_points=200,
                                    pharm_multi_vector=False, solvent=None)

In [ ]:
cond_pipe.evaluate(verbose=True, num_workers=1, mp_context='spawn')

In [ ]:
cond_pipe.evaluate(verbose=True, num_workers=8, mp_context='spawn')

In [ ]:
properties_df_cond, global_attr_cond = cond_pipe.to_pandas()

In [ ]:
global_attr_cond.head()

In [ ]:
properties_df_cond